In [1]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd


INPUT_PATH = "/content/final_dataset.json"
OUTPUT_DIR = Path("/content/split_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
DEV_RATIO = 0.20

LABELS = ["SUPPORTED", "NOT_ENOUGH_EVIDENCE", "REFUTED"]

random.seed(RANDOM_SEED)


with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

assert isinstance(data, list), "final_dataset.json phải là một JSON array/list."

print("Total claims:", len(data))

required_fields = ["claim_id", "chunk_id", "label", "claim_type"]
for item in data:
    for field in required_fields:
        if field not in item:
            raise ValueError(f"Missing field `{field}` in item: {item.get('claim_id')}")

# Check duplicate claim_id
claim_ids = [item["claim_id"] for item in data]
duplicated_claim_ids = [cid for cid, count in Counter(claim_ids).items() if count > 1]
assert len(duplicated_claim_ids) == 0, f"Duplicate claim_id found: {duplicated_claim_ids[:10]}"


# Group data by chunk_id
groups = defaultdict(list)
for item in data:
    groups[item["chunk_id"]].append(item)

group_list = list(groups.items())
random.shuffle(group_list)

total_n = len(data)
target_dev_n = round(total_n * DEV_RATIO)

total_label_count = Counter(item["label"] for item in data)
total_type_count = Counter(item["claim_type"] for item in data)

target_dev_label = {
    label: round(count * DEV_RATIO)
    for label, count in total_label_count.items()
}

target_dev_type = {
    ctype: round(count * DEV_RATIO)
    for ctype, count in total_type_count.items()
}

print("\nTarget dev size:", target_dev_n)
print("Target dev label:", target_dev_label)
print("Target dev claim_type:", target_dev_type)



def group_stats(items):
    return {
        "n": len(items),
        "label": Counter(x["label"] for x in items),
        "claim_type": Counter(x["claim_type"] for x in items),
    }


def score_if_add(current_n, current_label, current_type, group_items):
    """
    Lower score is better.
    Ưu tiên:
    1. Tổng size dev gần 20%
    2. Label gần target
    3. Claim_type gần target
    """
    stats = group_stats(group_items)

    new_n = current_n + stats["n"]
    new_label = current_label + stats["label"]
    new_type = current_type + stats["claim_type"]

    # Size penalty
    size_penalty = abs(new_n - target_dev_n) / max(target_dev_n, 1)

    # Label penalty
    label_penalty = 0
    for label, target in target_dev_label.items():
        label_penalty += abs(new_label.get(label, 0) - target) / max(target, 1)

    # Claim type penalty
    type_penalty = 0
    for ctype, target in target_dev_type.items():
        type_penalty += abs(new_type.get(ctype, 0) - target) / max(target, 1)

    # Label quan trọng hơn claim_type
    return (3.0 * size_penalty) + (5.0 * label_penalty) + (1.0 * type_penalty)


dev_chunk_ids = set()
test_chunk_ids = set()

current_dev_n = 0
current_dev_label = Counter()
current_dev_type = Counter()

# Sort groups: nhóm lớn xử lý trước để giảm lệch
group_list_sorted = sorted(
    group_list,
    key=lambda x: len(x[1]),
    reverse=True
)

for chunk_id, items in group_list_sorted:
    if current_dev_n >= target_dev_n:
        test_chunk_ids.add(chunk_id)
        continue

    score_add = score_if_add(
        current_dev_n,
        current_dev_label,
        current_dev_type,
        items
    )

    # Score nếu không add thì giữ nguyên dev hiện tại
    size_penalty_keep = abs(current_dev_n - target_dev_n) / max(target_dev_n, 1)

    label_penalty_keep = 0
    for label, target in target_dev_label.items():
        label_penalty_keep += abs(current_dev_label.get(label, 0) - target) / max(target, 1)

    type_penalty_keep = 0
    for ctype, target in target_dev_type.items():
        type_penalty_keep += abs(current_dev_type.get(ctype, 0) - target) / max(target, 1)

    score_keep = (3.0 * size_penalty_keep) + (5.0 * label_penalty_keep) + (1.0 * type_penalty_keep)

    # Add nếu giúp dev gần target hơn, hoặc dev còn quá nhỏ
    if score_add <= score_keep or current_dev_n < target_dev_n * 0.90:
        dev_chunk_ids.add(chunk_id)
        stats = group_stats(items)
        current_dev_n += stats["n"]
        current_dev_label += stats["label"]
        current_dev_type += stats["claim_type"]
    else:
        test_chunk_ids.add(chunk_id)

# Những group chưa được gán
all_chunk_ids = set(groups.keys())
assigned = dev_chunk_ids | test_chunk_ids
unassigned = all_chunk_ids - assigned
test_chunk_ids.update(unassigned)


# split data into dev and test
dev_data = []
test_data = []
full_with_split = []

for item in data:
    new_item = dict(item)

    if item["chunk_id"] in dev_chunk_ids:
        new_item["split"] = "dev"
        dev_data.append(new_item)
    elif item["chunk_id"] in test_chunk_ids:
        new_item["split"] = "test"
        test_data.append(new_item)
    else:
        raise RuntimeError("Có chunk_id chưa được gán split.")

    full_with_split.append(new_item)


# validation

assert len(dev_data) + len(test_data) == len(data)

dev_claim_ids = set(x["claim_id"] for x in dev_data)
test_claim_ids = set(x["claim_id"] for x in test_data)
assert dev_claim_ids.isdisjoint(test_claim_ids), "Leak claim_id giữa dev và test."

dev_chunks = set(x["chunk_id"] for x in dev_data)
test_chunks = set(x["chunk_id"] for x in test_data)
leaked_chunks = dev_chunks & test_chunks
assert len(leaked_chunks) == 0, f"Leak chunk_id giữa dev và test: {list(leaked_chunks)[:10]}"

print("SPLIT SUMMARY")
print("Dev claims:", len(dev_data), f"({len(dev_data) / len(data):.2%})")
print("Test claims:", len(test_data), f"({len(test_data) / len(data):.2%})")
print("Total:", len(dev_data) + len(test_data))

print("\nChunk leakage:", len(leaked_chunks))



# statistics
def make_count_df(items, field):
    counter = Counter(x[field] for x in items)
    return pd.DataFrame(
        sorted(counter.items()),
        columns=[field, "count"]
    )

label_dev = make_count_df(dev_data, "label").rename(columns={"count": "dev_count"})
label_test = make_count_df(test_data, "label").rename(columns={"count": "test_count"})
label_all = make_count_df(full_with_split, "label").rename(columns={"count": "total_count"})

label_stats = label_all.merge(label_dev, on="label", how="left").merge(label_test, on="label", how="left")
label_stats = label_stats.fillna(0)
label_stats["dev_ratio"] = label_stats["dev_count"] / label_stats["total_count"]
label_stats["test_ratio"] = label_stats["test_count"] / label_stats["total_count"]

type_dev = make_count_df(dev_data, "claim_type").rename(columns={"count": "dev_count"})
type_test = make_count_df(test_data, "claim_type").rename(columns={"count": "test_count"})
type_all = make_count_df(full_with_split, "claim_type").rename(columns={"count": "total_count"})

type_stats = type_all.merge(type_dev, on="claim_type", how="left").merge(type_test, on="claim_type", how="left")
type_stats = type_stats.fillna(0)
type_stats["dev_ratio"] = type_stats["dev_count"] / type_stats["total_count"]
type_stats["test_ratio"] = type_stats["test_count"] / type_stats["total_count"]

print("\nLABEL STATS")
display(label_stats)

print("\nCLAIM_TYPE STATS")
display(type_stats)


# save files
def save_json(path, items):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

def save_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_json(OUTPUT_DIR / "dev_dataset.json", dev_data)
save_json(OUTPUT_DIR / "test_dataset.json", test_data)
save_json(OUTPUT_DIR / "full_dataset_with_split.json", full_with_split)

save_jsonl(OUTPUT_DIR / "dev_dataset.jsonl", dev_data)
save_jsonl(OUTPUT_DIR / "test_dataset.jsonl", test_data)
save_jsonl(OUTPUT_DIR / "full_dataset_with_split.jsonl", full_with_split)

label_stats.to_csv(OUTPUT_DIR / "label_split_stats.csv", index=False, encoding="utf-8-sig")
type_stats.to_csv(OUTPUT_DIR / "claim_type_split_stats.csv", index=False, encoding="utf-8-sig")

print("\nSaved files to:", OUTPUT_DIR)
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

Total claims: 1311

Target dev size: 262
Target dev label: {'SUPPORTED': 87, 'NOT_ENOUGH_EVIDENCE': 87, 'REFUTED': 87}
Target dev claim_type: {'TIME_EVENT': 38, 'PERSON_ROLE': 61, 'GENEALOGY_SUCCESSION': 21, 'POLITICAL_ADMINISTRATIVE': 39, 'MILITARY_DIPLOMACY': 91, 'NATURAL_OMEN_DISASTER': 12}
SPLIT SUMMARY
Dev claims: 261 (19.91%)
Test claims: 1050 (80.09%)
Total: 1311

Chunk leakage: 0

LABEL STATS


,label,total_count,dev_count,test_count,dev_ratio,test_ratio
0,NOT_ENOUGH_EVIDENCE,437,87,350,0.199085,0.800915
1,REFUTED,437,87,350,0.199085,0.800915
2,SUPPORTED,437,87,350,0.199085,0.800915



CLAIM_TYPE STATS


,claim_type,total_count,dev_count,test_count,dev_ratio,test_ratio
0,GENEALOGY_SUCCESSION,103,22,81,0.213592,0.786408
1,MILITARY_DIPLOMACY,455,98,357,0.215385,0.784615
2,NATURAL_OMEN_DISASTER,61,13,48,0.213115,0.786885
3,PERSON_ROLE,307,52,255,0.169381,0.830619
4,POLITICAL_ADMINISTRATIVE,193,35,158,0.181347,0.818653
5,TIME_EVENT,192,41,151,0.213542,0.786458



Saved files to: /content/split_output
Files:
- claim_type_split_stats.csv
- dev_dataset.json
- dev_dataset.jsonl
- full_dataset_with_split.json
- full_dataset_with_split.jsonl
- label_split_stats.csv
- test_dataset.json
- test_dataset.jsonl


In [2]:
import shutil

zip_path = "/content/split_output.zip"
shutil.make_archive("/content/split_output", "zip", "/content/split_output")

print("Created:", zip_path)

Created: /content/split_output.zip


In [3]:
from google.colab import files
files.download("/content/split_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>